---
title: The phasic model hub
---

[`munch-group/phasic-traces`](https://github.com/munch-group/phasic-traces) is the public registry where pre-computed elimination artifacts are published. This notebook walks through the live registry — listing the artifacts available right now, downloading one, reusing it on a fresh local cache, and previewing what publishing a new model would look like.

Unlike the [reference tutorial](sharing.ipynb) (which uses a local mock registry so it can run offline), every cell here talks to the real GitHub repository. Re-execute the notebook to see the current state of the registry.

## Setup

In [1]:
import numpy as np
import phasic
from phasic.compute_repository import _param_compute_cache_dir

## What's published right now?

`phasic.list_computes()` downloads `registry.json` from the hub and returns one dict per published artifact. Use the `domain`, `model_type`, and `tags` keyword arguments to filter.

In [2]:
for entry in phasic.list_computes():
    print(
        f"{entry['compute_id']:24s}  "
        f"vertices={entry.get('vertices', '?'):>4}  "
        f"params={entry.get('param_length', '?')}  "
        f"hash={entry['graph_hash'][:16]}..."
    )

coal_n5_theta1            vertices=   6  params=1  hash=4d652ed73c66ed4a...
coal_n10_theta1           vertices=  11  params=1  hash=2c6488c6486ae2d2...
coal_n20_theta1           vertices=  21  params=1  hash=6f7690305bd5c9ec...


In [3]:
# Drill in: just the coalescent models, with their full metadata.
for entry in phasic.list_computes(model_type='coalescent'):
    print(entry['compute_id'])
    print(f"  description:    {entry['description']}")
    print(f"  tags:           {entry.get('tags', [])}")
    print(f"  format_revision: {entry['format_revision']}")
    print()

coal_n5_theta1
  description:    Kingman coalescent for n=5 haploid samples (1 parameter)
  tags:           ['coalescent', 'kingman', 'population-genetics']
  format_revision: 2

coal_n10_theta1
  description:    Kingman coalescent for n=10 haploid samples (1 parameter)
  tags:           ['coalescent', 'kingman', 'population-genetics']
  format_revision: 2

coal_n20_theta1
  description:    Kingman coalescent for n=20 haploid samples (1 parameter)
  tags:           ['coalescent', 'kingman', 'population-genetics']
  format_revision: 2



## Reusing a published model

If you build a graph that's structurally identical to one in the hub, `pull_cache()` downloads the pre-computed elimination and drops it into the local phasic cache. The next call to `expectation()`, `pdf()`, or `moments()` reads the file via the normal C-side cache lookup — no extra plumbing needed.

The hub has a coalescent for n=5 published as `coal_n5_theta1`. We'll recreate it locally and reuse the published elimination.

In [4]:
def coalescent_callback(state):
    """Standard Kingman coalescent: rate n(n-1)/2."""
    n = state[0]
    if n <= 1:
        return []
    return [(np.array([n - 1]), [n * (n - 1) / 2])]

g = phasic.Graph(coalescent_callback, ipv=[5])
print(f'vertices:      {g.vertices_length()}')
print(f'param_length:  {g.param_length()}')
print(f'graph hash:    {phasic.hash.compute_graph_hash(g).hash_hex[:32]}...')

vertices:      6
param_length:  1
graph hash:    4d652ed73c66ed4ada3dd13e0dde2051...


Wipe the local cache to simulate a fresh machine, then pull from the hub:

In [5]:
cache_dir = _param_compute_cache_dir()
hash_hex = phasic.hash.compute_graph_hash(g).hash_hex
bin_path = cache_dir / f'{hash_hex}.bin'

if bin_path.exists():
    bin_path.unlink()
print(f'before pull: file present = {bin_path.exists()}')

hit = g.pull_cache()
print(f'pull_cache():           {hit}')
print(f'after pull: file present = {bin_path.exists()}')
print(f'size: {bin_path.stat().st_size:,} bytes')

before pull: file present = False
pull_cache():           True
after pull: file present = True
size: 5,064 bytes


Now `expectation()` (and `pdf`, `moments`, etc.) reuses the downloaded elimination. The result must match what the publisher computed (1.6 = expected time to coalescence for n=5 under the standard rate):

In [6]:
exp = g.expectation()
print(f'expectation: {exp}')
assert abs(exp - 1.6) < 1e-10, 'unexpected expectation value'

expectation: 1.6


### Calling `pull_cache()` on an unregistered graph

`pull_cache()` returns `False` if there's no entry for the graph's content hash. It doesn't raise — you can check the return value to decide whether to run elimination locally or publish your own artifact afterwards.

In [7]:
# Build a coalescent with a non-standard sample size (n=4 is not
# published as of this writing).
g_unpublished = phasic.Graph(coalescent_callback, ipv=[4])
hit = g_unpublished.pull_cache()
print(f'pull_cache for n=4: {hit}')

pull_cache for n=4: False


## Verifying a published model end-to-end

One use of the hub: verify that someone else's published expectation matches what *you* get when running the elimination locally from the same callback. This is roughly the workflow for reviewing a paper's numerical claims.

Clear the cache, recompute locally, and compare against the published artifact's result.

In [8]:
# Step 1: clear local cache, compute fresh.
if bin_path.exists():
    bin_path.unlink()
g_fresh = phasic.Graph(coalescent_callback, ipv=[5])
exp_local = g_fresh.expectation()
print(f'local computation:  {exp_local}')

# Step 2: clear cache again, pull from hub.
if bin_path.exists():
    bin_path.unlink()
g_hub = phasic.Graph(coalescent_callback, ipv=[5])
assert g_hub.pull_cache() is True
exp_hub = g_hub.expectation()
print(f'hub artifact:       {exp_hub}')

print(f'agreement: {abs(exp_local - exp_hub) < 1e-10}')

local computation:  1.6
hub artifact:       1.6
agreement: True


Both come out to 1.6 to machine precision. The hub artifact is a faithful capture of the local elimination.

## Previewing what a publish would look like

`g.push_cache(..., dry_run=True)` builds the artifact, computes its SHA-256, and returns a JSON string of the entry that *would* be added to `registry.json`. It does not touch the network. Use this to check your metadata before opening an actual PR.

We'll preview what publishing a small custom model would generate:

In [ ]:
# A two-step parameterised chain (not in the hub).
g_demo = phasic.Graph(1)
v0 = g_demo.starting_vertex()
v1 = g_demo.find_or_create_vertex([1])
v2 = g_demo.find_or_create_vertex([2])
v0.add_edge(v1, [1.0])
v1.add_edge(v2, [1.0])
g_demo.update_weights([2.0])

entry_json = g_demo.push_cache(
    id='demo_two_step_chain',
    description='Two-step parameterised chain (demo, not for publishing)',
    domain='examples',
    model_type='chain',
    tags=['demo'],
    dry_run=True,
)
print(entry_json)

PTDBackendError: command failed (exit 1): gh pr create --title Add compute artifact: demo_two_step_chain --body Adds the C-elimination compute artifact for `demo_two_step_chain`.

- graph_hash: `29dc136099b6cf8de1d9180a234a2358b63c5ddca6649889f9ed6a25d914418c`
- format_revision: 2
- vertices: 3
stdout: 
stderr: pull request create failed: GraphQL: Resource not accessible by personal access token (createPullRequest)


Inspect the preview before deciding whether to publish for real. A few things worth checking:

- **`graph_hash`** matches what `compute_graph_hash(g)` returned. Anyone re-building the same graph will get the same hash and reach this artifact via `pull_cache()`.
- **`format_revision`** is `2` (the current C-side format). Older builds with `format_revision < 2` would reject this entry; newer builds will load it.
- **`parent.sha256`** is the cryptographic checksum of the `.bin`. Consumers verify it before installing.
- **`metadata.vertices`** and **`param_length`** are filled in for you.
- **`metadata.author`** comes from your local `git config user.name/email`. Override with the `author=` keyword if you want to attribute differently.

Once the preview looks right, drop `dry_run=True` to actually open the PR (requires `gh auth login` set up first).

## When `pull_cache()` is useful in practice

- **CI pipelines**: every workflow run starts with an empty cache. Without `pull_cache`, every CI job re-eliminates the same graph. With it, the first job published an artifact and all subsequent jobs share it.
- **Cluster jobs**: SLURM tasks on different nodes don't share a local `~/.phasic_cache/`. Calling `pull_cache()` at the start of the job lets each node download once instead of eliminating once.
- **Paper reproducibility**: publish artifacts for the exact models in a paper, link the hub entry in the paper, readers re-compute by `Graph(callback); g.pull_cache(); g.expectation()`.
- **Group workflows**: a lab member runs `push_cache()` once for a shared model; everyone else's `pull_cache()` calls hit instantly.

## Summary

| What | API |
|------|-----|
| What's published? | `phasic.list_computes(...)` |
| Reuse a published model | `g.pull_cache()` |
| Check before publishing | `g.push_cache(..., dry_run=True)` |
| Publish for real | `g.push_cache(...)` (needs `gh auth login`) |

For the full reference, see [the sharing tutorial](sharing.ipynb).